# Optimization Methods — AI Lab Instructor Notebook

*Deep Learning · Hard*



6 tasks, 1 left as exercises (no source solution).

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import math
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Optimization Methods — Student Lab

Focus: implement SGD, Momentum, and Adam; compare convergence on ill-conditioned and nonconvex objectives.

### Optimization landscape intuition
![Convex vs non-convex objective intuition](/images/w7-d2-convex-vs-nonconvex.svg)

In [ ]:
# No source solution — left as an exercise.

## Task 2: Implement `make_quadratic(d, cond)` returning (A, f, grad).
*Section: Ill-conditioned quadratic objective*

## Section 1 — Ill-conditioned quadratic objective

Top (contour) view of a convex paraboloid (ill-conditioned):

![Top contour view of a convex paraboloid (ill-conditioned)](/images/w7-d2-ill-conditioned-original.png)

We optimize: f(x) = 0.5 * x^T A x where A is SPD with large condition number.

### Why gradient descent struggles in ill-conditioned problems
Instead of moving straight to the minimum, gradient descent often zig-zags:

```text
      ↘
       ↘
        ↘
       ↗
      ↘
     ↗
    ↘
```

Reason: one direction is steep while the other is flat, so each update over-corrects across the valley and makes slow progress along the flat direction.

![Gradient descent zig-zag path in an ill-conditioned valley](/images/w7-d2-zigzag-descent.svg)

### Task 1.1
Implement `make_quadratic(d, cond)` returning (A, f, grad).

### Task 1.2
Gradient check with finite differences.

In [ ]:
def make_quadratic(d: int, cond: float, seed=0):
    r = np.random.default_rng(seed)
    M = r.normal(size=(d, d))
    Q, _ = np.linalg.qr(M)
    eigs = np.geomspace(1.0, cond, num=d)
    A = Q @ np.diag(eigs) @ Q.T

    def f(x):
        return 0.5 * float(x.T @ A @ x)
    def g(x):
        return A @ x
    return A, f, g

def finite_diff_grad(f, x, eps=1e-6):
    g = np.zeros_like(x)
    for i in range(len(x)):
        xp = x.copy(); xp[i] += eps
        xm = x.copy(); xm[i] -= eps
        g[i] = (f(xp) - f(xm)) / (2 * eps)
    return g

d = 20
A, f, grad = make_quadratic(d=d, cond=1e4, seed=0)
x0 = rng.normal(size=d)
g_fd = finite_diff_grad(f, x0)
g_an = grad(x0)
err = float(np.linalg.norm(g_fd - g_an) / (np.linalg.norm(g_fd) + 1e-12))
print('rel_grad_error', err)
check('grad_check', err < 1e-5)

In [ ]:
# Checks
check('grad_check', err < 1e-5)

**Why this works.** ## Section 1 — Ill-conditioned quadratic

Rationale: this is the cleanest way to see ill-conditioning: gradients point mostly along steep directions, forcing tiny steps that barely move along flat directions.

## Task 3: Implement:
*Section: Implement optimizers*

## Section 2 — Implement optimizers

We will optimize vectors `w` with updates based on gradients.

### Task 2.1
Implement:
- SGD
- Momentum
- Adam (with bias correction)

In [ ]:
def sgd_step(w, g, lr):
    return w - lr * g

def momentum_init(w):
    return {'v': np.zeros_like(w)}

def momentum_step(w, g, state, lr, beta=0.9):
    state['v'] = beta * state['v'] + (1 - beta) * g
    return w - lr * state['v'], state

def adam_init(w):
    return {'m': np.zeros_like(w), 'v': np.zeros_like(w), 't': 0}

def adam_step(w, g, state, lr, b1=0.9, b2=0.999, eps=1e-8):
    state['t'] += 1
    t = state['t']
    state['m'] = b1 * state['m'] + (1 - b1) * g
    state['v'] = b2 * state['v'] + (1 - b2) * (g * g)
    mhat = state['m'] / (1 - b1**t)
    vhat = state['v'] / (1 - b2**t)
    w = w - lr * mhat / (np.sqrt(vhat) + eps)
    return w, state

w = rng.normal(size=5)
g = rng.normal(size=5)
check('sgd_shape', sgd_step(w, g, 0.1).shape == w.shape)
wm, st = momentum_step(w, g, momentum_init(w), 0.1)
check('mom_shape', wm.shape == w.shape)
wa, st2 = adam_step(w, g, adam_init(w), 0.1)
check('adam_shape', wa.shape == w.shape)

In [ ]:
# Checks
check('sgd_shape', sgd_step(w, g, 0.1).shape == w.shape)
check('mom_shape', wm.shape == w.shape)
check('adam_shape', wa.shape == w.shape)

**Why this works.** ## Section 2 — Optimizers

Rationale:
- Momentum is an EMA of gradients (low-pass filter) and helps traverse narrow valleys.
- Adam adds per-parameter scaling via second-moment estimates + bias correction.

## Task 4: Write `run_opt(optimizer_name, steps, lr, ...)` and compare optimizer dynamics.
*Section: Compare convergence on the quadratic*

## Section 3 — Compare convergence on the quadratic

### Task 3.1
Write `run_opt(optimizer_name, steps, lr, ...)` and compare optimizer dynamics from the same starting point.

Track more than just objective decrease:
- objective value
- gradient norm
- update norm
- a simple trajectory slice using the first two coordinates

Optimization is about **how** the parameter vector moves, not only whether the loss goes down.

In [ ]:
$$10

In [ ]:
# Checks
h_sgd = run_opt('sgd', x0, f, grad, 50, 1e-4)[1]
h_mom = run_opt('mom', x0, f, grad, 50, 2e-4)[1]
h_adam = run_opt('adam', x0, f, grad, 50, 5e-3)[1]
check('same_start', np.isclose(h_sgd['obj'][0], h_mom['obj'][0]) and np.isclose(h_mom['obj'][0], h_adam['obj'][0]))
check('hist_lengths', len(h_sgd['obj']) == 51 and len(h_sgd['grad_norm']) == 50 and len(h_sgd['update_norm']) == 50 and len(h_sgd['traj']) == 51)
check('all_decrease_somewhat', h_sgd['obj'][0] > h_sgd['obj'][-1] and h_mom['obj'][0] > h_mom['obj'][-1] and h_adam['obj'][0] > h_adam['obj'][-1])

**Why this works.** ## Section 3 — Convergence comparison

Rationale: the objective curve is only one view. Optimization dynamics also include:
- **gradient norm**: how steep the local landscape is
- **update norm**: how aggressively the optimizer moves parameters
- **trajectory**: whether updates zig-zag, smooth out, or rescale directions

Momentum should reduce zig-zagging in narrow valleys. Adam may take very different sized coordinate-wise steps because it rescales each coordinate by its estimated second moment.

## Task 5: Implement LR schedules and compare schedule dynamics.
*Section: Learning-rate schedules*

## Section 4 — Learning-rate schedules

### Task 4.1
Implement:
- constant LR
- step decay
- cosine decay

Run them with SGD and compare not just final objective, but also:
- average learning rate
- average update norm
- how much the last 50 steps improve relative to the first 50

In [ ]:
$$12

**Why this works.** ## Section 4 — Learning-rate schedules

Rationale: schedules change the *optimization regime* over time. Large early steps help move fast; smaller late steps help settle. Looking at update norms and early-vs-late progress makes that visible.

## Task 6: Implement gradient clipping by norm and show it stabilizes updates.
*Section: Gradient clipping (nonconvex)*

## Section 5 — Gradient clipping (nonconvex)

We use a simple nonconvex function with large gradients far from the origin.

### Non-convex saddle intuition
![Saddle-point contour intuition](/images/w7-d2-saddle-point.svg)

Near a saddle, gradient magnitude can be small but curvature is mixed, which can stall or destabilize updates.

### Task 5.1
Implement gradient clipping by norm and show it stabilizes updates.

Use intentionally aggressive settings so clipping actually matters.

Track:
- raw gradient norm
- clipped gradient norm actually used for the update
- update norm
- how many steps were clipped
- whether the run diverged

In [ ]:
$$13

In [ ]:
# Checks
check('clip_caps_used_grad', float(max(h2['used_grad_norm'])) <= 0.5 + 1e-8)
check('clipping_activated', h2['n_clipped'] > 0)
check('clip_stabilizes', h1['diverged'] and (not h2['diverged']))

**Why this works.** ## Section 5 — Gradient clipping

Rationale: clipping is an optimization stability tool, not just a loss trick. In this stronger setup, the unclipped run takes an excessively large step and diverges, while the clipped run caps the gradient norm and stays stable.

The key evidence is:
- **raw vs used gradient norm**: shows clipping actually changed the update
- **n_clipped**: shows clipping activated on real steps
- **diverged flag**: shows why clipping matters operationally